# PYNQ-Z1 DIANSAI2 AD9767 Bring-up

Pure AD9767 overlay: board IO plus dual-channel AD9767 output. AD9102 and AD9226 are not present in this bitstream.


In [ ]:
# Cell 1: load overlay and bind fixed MMIO addresses.
from time import sleep, time
from pynq import Overlay, MMIO

BITFILE = 'base_add.bit'
overlay = Overlay(BITFILE)

LED_BASE = 0x40000000
AD9767_BASE = 0x40001000

led_ip = MMIO(LED_BASE, 0x1000)
ad9767_ip = MMIO(AD9767_BASE, 0x1000)

LED_CTRL = 0x00
LED_VALUE = 0x08
LED_STATUS = 0x0C

COLOR_RGB = {
    'OFF': 0b000,
    'RED': 0b001,
    'GREEN': 0b010,
    'BLUE': 0b100,
    'YELLOW': 0b011,
    'MAGENTA': 0b101,
    'PURPLE': 0b101,
    'CYAN': 0b110,
    'WHITE': 0b111,
}

def set_board_io(led_mask=0, ld4_color=0, ld5_color=0):
    value = (int(led_mask) & 0xF) | ((int(ld5_color) & 0x7) << 4) | ((int(ld4_color) & 0x7) << 7)
    led_ip.write(LED_CTRL, 0x00)
    led_ip.write(LED_VALUE, value)
    return value

def set_leds(ld0=False, ld1=False, ld2=False, ld3=False, ld4='OFF', ld5='OFF'):
    led_mask = (1 if ld0 else 0) | (2 if ld1 else 0) | (4 if ld2 else 0) | (8 if ld3 else 0)
    return set_board_io(led_mask, COLOR_RGB[str(ld4).upper()], COLOR_RGB[str(ld5).upper()])

def read_board_io():
    status = led_ip.read(LED_STATUS)
    return {
        'raw': status,
        'led_mask': status & 0xF,
        'rgb_raw': (status >> 4) & 0x3F,
        'buttons_raw': (status >> 10) & 0xF,
    }

def read_buttons():
    raw = (led_ip.read(LED_STATUS) >> 10) & 0xF
    return {'raw': raw, 'btn0': raw & 1, 'btn1': (raw >> 1) & 1, 'btn2': (raw >> 2) & 1, 'btn3': (raw >> 3) & 1}

print('Overlay loaded:', BITFILE)
print('IP dict keys:', sorted(overlay.ip_dict.keys()))
print('LED status:', read_board_io())
print('AD9767 version=0x%08X sample_rate=%d' % (ad9767_ip.read(0x38), ad9767_ip.read(0x3C)))


In [ ]:
# Cell 2: test physical LEDs, RGB LEDs, and buttons.
print('Walking LD0..LD3')
for phys in range(4):
    kwargs = {'ld0': False, 'ld1': False, 'ld2': False, 'ld3': False}
    kwargs['ld%d' % phys] = True
    value = set_leds(**kwargs)
    sleep(0.4)
    print('LD%d on, write=0x%03X status=%s' % (phys, value, read_board_io()))

print('\nRGB quick test')
for ld4, ld5 in [('RED', 'OFF'), ('GREEN', 'OFF'), ('BLUE', 'OFF'), ('OFF', 'RED'), ('OFF', 'GREEN'), ('OFF', 'BLUE'), ('BLUE', 'GREEN')]:
    value = set_leds(ld4=ld4, ld5=ld5)
    sleep(0.4)
    print('LD4=%-5s LD5=%-5s write=0x%03X' % (ld4, ld5, value))

print('\nLive button monitor for 10 seconds. Press BTN0..BTN3 one by one.')
t_end = time() + 10.0
last = None
while time() < t_end:
    buttons = read_buttons()
    if buttons['raw'] != last:
        print('raw=0b%s BTN0=%d BTN1=%d BTN2=%d BTN3=%d' % (
            format(buttons['raw'], '04b'), buttons['btn0'], buttons['btn1'], buttons['btn2'], buttons['btn3']))
        last = buttons['raw']
    sleep(0.03)

set_leds(False, False, False, False, ld4='OFF', ld5='OFF')
print('Board IO test done.')


In [ ]:
# Cell 3: AD9767 C-problem wireless signal parameter test.
# Provisional output selection:
#   DAC A can output SD/SM/SOUT/DC
#   DAC B can output SD/SM/SOUT/DC/MOD_SQUARE/MOD_SINE
# First use an oscilloscope on DAC A/B and switch out_a/out_b below.
from pynqz1_diansai2_ad9767 import AD9767Signal

ad9767 = AD9767Signal(base_addr=AD9767_BASE)

cfg = ad9767.configure_wireless(
    carrier_hz=30_000_000,      # C problem: 30-40 MHz, 1 MHz step
    mode='AM',                  # 'CW' or 'AM'
    sd_vrms=0.25,               # keep AM headroom first, then increase
    sd_phase_deg=0,
    am_depth_percent=30,        # start low; high depth needs lower carrier gain
    sm_delay_ns=80,             # target range 50-200 ns
    sm_phase_deg=30,            # target range 0-180 deg
    sm_atten_db=6,              # target range 0-20 dB
    out_a='SD',                 # SD, SM, SOUT, DC
    out_b='MOD_SQUARE',         # SD, SM, SOUT, DC, MOD_SQUARE, MOD_SINE
)

print('AD9767 config:')
for k, v in cfg.items():
    print('  %-22s = %s' % (k, v))
print('AD9767 status:', ad9767.status())


In [ ]:
# Cell 4: common C-problem presets.
# Run one line at a time when tuning with the oscilloscope.

# 35 MHz CW, show direct and multipath
# ad9767.configure_wireless(carrier_hz=35_000_000, mode='CW', sd_vrms=0.5, sm_delay_ns=80, sm_phase_deg=30, sm_atten_db=6, out_a='SD', out_b='SM')

# 35 MHz AM, 2 MHz modulation, show direct and summed output
# ad9767.configure_wireless(carrier_hz=40_000_000, mode='AM', sd_vrms=0.25, am_depth_percent=30, sm_delay_ns=80, sm_phase_deg=30, sm_atten_db=6, out_a='SD', out_b='MOD_SQUARE')

# Sweep manual carrier points: 30, 35, 40 MHz
# for fc in [30_000_000, 35_000_000, 40_000_000]:
#     cfg = ad9767.configure_wireless(carrier_hz=fc, mode='CW', sd_vrms=0.5, out_a='SD', out_b='SOUT')
#     print(fc, cfg)
#     sleep(1.0)

print('Edit and run a preset above.')
